<h1 style="color: #1F4E78; font-weight: 700;">SupplySight AI</h1>

<h1 style="color: #1F4E78; font-weight: 700;">Enterprise Procurement Decision Intelligence Platform</h1>

<h1 style="color: #1F4E78; font-weight: 700;">Notebook 01A</h1>

<h1 style="color: #1F4E78; font-weight: 700;">⛽ External Data ETL: Fuel Prices Acquisition</h1>

<h2 style="color: #1F4E78; font-weight: 700;">Executive Summary</h2>

This notebook focuses on acquiring historical fuel price data from the U.S. Energy Information Administration (EIA). Fuel prices serve as a critical proxy for transportation and logistics costs, which directly impact procurement expenditure.

Unlike other external datasets such as commodities, fuel price data will later be transformed into a continuous daily time series to align with daily procurement transaction analysis, enabling more granular insights into cost fluctuations.

---

<h2 style="color: #1F4E78; font-weight: 700;">Objective</h2>

The primary objectives of this notebook are:

* Retrieve historical diesel fuel prices from the EIA API
* Standardize the dataset for consistency and downstream processing
* Preserve temporal integrity of the data
* Prepare the dataset for daily expansion during the data cleaning phase

---

<h2 style="color: #1F4E78; font-weight: 700;">Business Rationale</h2>

Transportation costs represent a significant portion of procurement expenses. Fuel price volatility directly affects:

* Shipping and freight expenses
* Distribution and logistics costs
* Supplier transportation charges
* Overall procurement spend

By integrating fuel price data into the analytics pipeline, procurement performance can be evaluated in the context of changing logistics costs, enabling more informed and proactive decision-making.

---

<h2 style="color: #1F4E78; font-weight: 700;">Data Source</h2>

* **Provider:** U.S. Energy Information Administration (EIA)
* **Fuel Type:** No. 2 Diesel
* **Source Frequency:** Weekly
* **Project Output:** Daily fuel price series (post preprocessing)

---

<h2 style="color: #1F4E78; font-weight: 700;">ETL Workflow</h2>

```
EIA API
   ↓
Historical Fuel Prices
   ↓
Standardization
   ↓
Validation
   ↓
Daily Expansion (03B)
```

---



<h1 style="color: #1F4E78; font-weight: 700;">Step 1: Fetch Weekly Fuel Price Data from the U.S. Energy Information Administration (EIA) API</h1>

<h2 style="color: #1F4E78; font-weight: 700;">Objective</h2>

Retrieve historical weekly diesel fuel price data from the U.S. Energy Information Administration (EIA) API and store it as a CSV file for further analysis.

<h2 style="color: #1F4E78; font-weight: 700;">What this code does</h2>

- Imports the required Python libraries:
  - **requests** for making API requests.
  - **pandas** for data manipulation.
  - **os** for file and directory management.
- Creates the output directory if it does not already exist.
- Defines the EIA API key required to access the dataset.
- Constructs the API request URL with the required query parameters.
- Sends a GET request to the EIA API.
- Verifies that the request was successful.
- Converts the returned JSON response into a Pandas DataFrame.
- Displays the total number of records and a preview of the dataset.
- Saves the raw fuel price data as a CSV file for future preprocessing.

<h2 style="color: #1F4E78; font-weight: 700;">Dataset Information</h2>

The retrieved dataset contains weekly diesel fuel prices published by the U.S. Energy Information Administration (EIA). Each record includes:
- Date (reporting period)
- Geographic region
- Fuel product type
- Fuel price
- Measurement unit

<h2 style="color: #1F4E78; font-weight: 700;">Output</h2>
A raw dataset named **fuel_prices.csv** is created in the `data/external/` directory, serving as the source data for subsequent cleaning and analysis.

In [6]:
import warnings
warnings.filterwarnings("ignore")

import requests
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [7]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\chavi\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


API_KEY = "oA2fgLckNgDvg6BjDRDPaAfyDFEQbsWyve7kjVNQ"

BASE_URL = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("FUEL_API_KEY")

if not API_KEY:
    raise ValueError("FUEL_API_KEY is missing. Check your .env file.")

print("API key loaded:", bool(API_KEY))

API key loaded: True


In [24]:
BASE_URL = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"

In [25]:
params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[product][]": "EPD2D",
    "start": "2015-01-01",
    "end": "2018-12-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}

In [26]:
response = requests.get(
    BASE_URL,
    params=params
)
print("Status Code :", response.status_code)

Status Code : 200


In [27]:
json_data = response.json()

print(json_data.keys())

dict_keys(['response', 'request', 'apiVersion', 'ExcelAddInVersion'])


In [28]:
fuel = pd.DataFrame(
    json_data["response"]["data"]
)

fuel.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2015-01-05,R20,PADD 2,EPD2D,No 2 Diesel,PTE,Retail Sales,EMD_EPD2D_PTE_R20_DPG,Midwest No 2 Diesel Retail Prices (Dollars per...,3.102,$/GAL
1,2015-01-05,R5XCA,PADD 5 EXCEPT CALIFORNIA,EPD2D,No 2 Diesel,PTE,Retail Sales,EMD_EPD2D_PTE_R5XCA_DPG,West Coast (PADD 5) Except California No 2 Die...,3.074,$/GAL
2,2015-01-05,R1Z,PADD 1C,EPD2D,No 2 Diesel,PTE,Retail Sales,EMD_EPD2D_PTE_R1Z_DPG,Lower Atlantic (PADD 1C) No 2 Diesel Retail Pr...,3.109,$/GAL
3,2015-01-05,R30,PADD 3,EPD2D,No 2 Diesel,PTE,Retail Sales,EMD_EPD2D_PTE_R30_DPG,Gulf Coast No 2 Diesel Retail Prices (Dollars ...,3.045,$/GAL
4,2015-01-05,R50,PADD 5,EPD2D,No 2 Diesel,PTE,Retail Sales,EMD_EPD2D_PTE_R50_DPG,West Coast No 2 Diesel Retail Prices (Dollars ...,3.22,$/GAL


In [29]:
fuel.columns

Index(['period', 'duoarea', 'area-name', 'product', 'product-name', 'process',
       'process-name', 'series', 'series-description', 'value', 'units'],
      dtype='str')


---

<h1 style="color: #1F4E78; font-weight: 700;">Step 2: Clean and Prepare the Fuel Price Dataset</h1>

<h2 style="color: #1F4E78; font-weight: 700;">Objective</h2>
Extract only the relevant features from the raw dataset, rename the columns for improved readability, and convert the date column into an appropriate datetime format.

<h2 style="color: #1F4E78; font-weight: 700;">What this code does</h2>

- Selects only the columns required for analysis.
- Removes unnecessary information from the original dataset.
- Renames columns using descriptive and analysis-friendly names:
  - `period` → `Date`
  - `area-name` → `Region`
  - `product-name` → `Fuel_Type`
  - `value` → `Fuel_Price`
  - `units` → `Unit`
- Converts the `Date` column into a Pandas datetime object to support time-series analysis.
- Displays the first few rows of the cleaned dataset.

<h2 style="color: #1F4E78; font-weight: 700;">Why this step is important</h2>

Data preprocessing improves readability, consistency, and usability. Converting dates into datetime format enables filtering, sorting, grouping, and trend analysis over time.

<h2 style="color: #1F4E78; font-weight: 700;">Output</h2>

A cleaned DataFrame named **fuel_df** containing only the relevant variables required for downstream analysis and visualization.

In [31]:
fuel_raw = pd.DataFrame(json_data["response"]["data"])

fuel = fuel_raw[
    [
        "period",
        "area-name",
        "product-name",
        "value",
        "units"
    ]
].copy()

fuel.rename(
    columns={
        "period": "Date",
        "area-name": "Region",
        "product-name": "Fuel_Type",
        "value": "Fuel_Price",
        "units": "Unit"
    },
    inplace=True
)

fuel["Date"] = pd.to_datetime(fuel["Date"])

fuel.head()

,Date,Region,Fuel_Type,Fuel_Price,Unit
0,2015-01-05,PADD 2,No 2 Diesel,3.102,$/GAL
1,2015-01-05,PADD 5 EXCEPT CALIFORNIA,No 2 Diesel,3.074,$/GAL
2,2015-01-05,PADD 1C,No 2 Diesel,3.109,$/GAL
3,2015-01-05,PADD 3,No 2 Diesel,3.045,$/GAL
4,2015-01-05,PADD 5,No 2 Diesel,3.22,$/GAL


In [10]:
print("="*70)
print("FUEL PRICE DATASET SUMMARY")
print("="*70)

print(f"Rows              : {fuel.shape[0]}")
print(f"Columns           : {fuel.shape[1]}")

print()

print("Date Range")
print(fuel["period"].min(), "to", fuel["period"].max())

print()

print("Duplicate Rows :", fuel.duplicated().sum())

print()

print("Missing Values")
print(fuel.isnull().sum())

FUEL PRICE DATASET SUMMARY
Rows              : 2299
Columns           : 11

Date Range
2015-01-05 to 2018-12-31

Duplicate Rows : 0

Missing Values
period                0
duoarea               0
area-name             0
product               0
product-name          0
process               0
process-name          0
series                0
series-description    0
value                 0
units                 0
dtype: int64


<h1 style="color: #1F4E78; font-weight: 700;">Project Paths</h1>

In [32]:
PROJECT_PATH = Path(r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform")

RAW = PROJECT_PATH / "Datasets" / "Raw Datasets"

EXTERNAL = PROJECT_PATH / "Datasets" / "External Datasets"

PROCESSED = PROJECT_PATH / "Datasets" / "Processed Datasets"

PROCESSED.mkdir(exist_ok=True)

In [33]:
fuel.to_csv(
    EXTERNAL / "fuel_prices.csv",
    index=False
)